# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mah-gie/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row represents one unique web page (URL).

**Time Window:** A mid-panel historical month, specifically March 2026 (2026-03), to ensure we don't accidentally train on the final test month.

In [1]:
import duckdb
from google.colab import userdata
import pandas as pd

# 1. Grab your token
hf_token = userdata.get('HF_TOKEN')

# 2. Open a DuckDB connection and hand it your Hugging Face token
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

print("Reaching into the warehouse specifically for March 2026...")

# 3. Use DuckDB to query ONLY the March data directly from the cloud
query = """
SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
"""

# 4. Convert just that small slice into our Pandas dataframe
df_march = con.sql(query).df()

print(f"Time window isolated: March 2026.")
print(f"Total rows in this window: {len(df_march)}")

Reaching into the warehouse specifically for March 2026...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Time window isolated: March 2026.
Total rows in this window: 9841378


## 2. Fields: feature / label / context / excluded

* **Features:** `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`.

* **Label (Target):** `is_declining` (Target).

* **Context:** `content_hash_id` (Identifies the specific page).

* **Excluded:** Label-derived metrics to deliberately prevent data leakage.

In [6]:
features = ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']
label = 'is_declining'
context = 'content_hash_id'
excluded = ['leakage_proxy']

print(f"Model will train on {len(features)} safe features.")

Model will train on 4 safe features.


## 3. Verify it with queries (grain, counts, missing values, windows)


In [7]:
# 1. Grain Check
print(f"Total Rows: {len(df_march)}")
print(f"Unique Content IDs: {df_march['content_hash_id'].nunique()}")
print("Note: The grain is daily performance per unique content hash.")

# 2. Availability Check (The 'IS TRUE' assignment requirement)
available_rows = df_march[df_march['gsc_data_available'] == True]
print(f"Rows with GSC data available (IS TRUE): {len(available_rows)}")

# 3. Missing Values Check
print("\n--- Missing Values Check ---")
print(df_march[features].isna().sum())

Total Rows: 9841378
Unique Content IDs: 331437
Note: The grain is daily performance per unique content hash.
Rows with GSC data available (IS TRUE): 3611061

--- Missing Values Check ---
client_has_gsc              0
client_has_ga4              0
gsc_data_available          0
ga4_data_available    3018741
dtype: int64


## 4. Data limits

**What this data cannot tell us:**

This dataset relies purely on historical Google Search Console metrics. It cannot tell us why a page is declining (e.g., if a competitor published a better article, or if the search intent changed). It also completely misses traffic drops from other sources like social media or direct links. It is purely an indicator of search decay.

In [8]:
print("Data limits conceptually documented in the markdown cell above.")

Data limits conceptually documented in the markdown cell above.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.